# Waveform Detection Radar Lab (STUDENT)

**Topic:** Waveform detection (matched filtering) + Neyman–Pearson thresholding + ROC curves + range estimation (monostatic radar).

**Learning goals**
1. Implement a matched filter detector for a known waveform.
2. Set a detection threshold from a desired **false alarm probability** $P_{FA}$.
3. Estimate **ROC curves** empirically via Monte Carlo simulations.
4. Estimate **range** from the time delay of the matched-filter peak.

**Model**

$$
\begin{aligned}
H_0 &: x[n] = w[n] \\
H_1 &: x[n] = A\,s[n-n_0] + w[n]
\end{aligned}
$$

where $w[n]\sim\mathcal{N}(0,\sigma^2)$ i.i.d., $s[n]$ is the known transmitted waveform, $A$ is the echo amplitude, and $n_0$ is the (unknown) delay related to range.

**Matched filter statistic (known delay)**

$$
T = \sum_n x[n]\,s[n-n_0]
$$

and the Neyman–Pearson decision rule is

$$
T \mathop{\gtrless}_{H_0}^{H_1} \gamma.
$$

**Notes**
- This notebook is self-contained and uses synthetic data (no hardware).
- Fill the TODO blocks and answer the interpretation questions.

In [8]:
# Imports
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def db2lin(db):
    return 10.0**(db/10.0)

def lin2db(lin):
    return 10.0*np.log10(lin)

def qfunc(x):
    # Q(x) = 0.5*erfc(x/sqrt(2)) = 0.5*(1 - erf(x/sqrt(2)))
    from math import erf
    z = np.asarray(x, dtype=float)/np.sqrt(2.0)
    return 0.5*(1.0 - np.vectorize(erf)(z))

def qinv(p):
    # Numerical inverse of Q-function.
    # If SciPy is installed: uses erfcinv.
    # Else: uses robust bisection.
    p = np.asarray(p, dtype=float)
    try:
        from scipy.special import erfcinv
        # Q(x)=0.5*erfc(x/sqrt(2)) => x = sqrt(2)*erfcinv(2p)
        return np.sqrt(2.0) * erfcinv(2.0*p)
    except Exception:
        lo = -20.0*np.ones_like(p)
        hi =  20.0*np.ones_like(p)
        for _ in range(90):
            mid = 0.5*(lo+hi)
            val = qfunc(mid)
            lo = np.where(val > p, mid, lo)   # need larger x
            hi = np.where(val <= p, mid, hi)
        return 0.5*(lo+hi)

def plot_signals(x0, x1=None, x_axis=None):
    """
    Plot one or two signals using Plotly.

    Parameters
    ----------
    x0 : array-like
        First signal (mandatory).
    x1 : array-like or None, optional
        Second signal. If None, only one plot is generated.
    x_axis : array-like or None, optional
        Common x-axis. If None, sample index starting at 0 is used.

    Notes
    -----
    If x1 is provided, x0 and x1 must have the same length.
    """

    n = len(x0)

    if x_axis is None:
        x_axis = list(range(n))
    else:
        if len(x_axis) != n:
            raise ValueError("x_axis must have the same length as the signal(s)")

    if x1 is None:
        # Single signal
        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=x0,
                mode="lines",
                name="x0"
            )
        )

        fig.update_xaxes(title_text="Sample index")
        fig.update_yaxes(title_text="Amplitude")

        fig.update_layout(
            height=400,
            title_text="Signal representation",
            margin=dict(l=60, r=20, t=60, b=60)
        )

    else:
        # Two signals
        if len(x1) != n:
            raise ValueError("x0 and x1 must have the same length")

        fig = make_subplots(
            rows=2,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            subplot_titles=("Signal x0", "Signal x1")
        )

        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=x0,
                mode="lines",
                name="x0"
            ),
            row=1,
            col=1
        )

        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=x1,
                mode="lines",
                name="x1"
            ),
            row=2,
            col=1
        )

        fig.update_xaxes(title_text="Sample index", row=2, col=1)
        fig.update_yaxes(title_text="Amplitude", row=1, col=1)
        fig.update_yaxes(title_text="Amplitude", row=2, col=1)

        fig.update_layout(
            height=600,
            title_text="Signal representation",
            showlegend=False,
            margin=dict(l=60, r=20, t=60, b=60)
        )

    fig.show()


## 1) Radar and waveform parameters

We simulate a monostatic radar:
- speed of light $c$
- sampling frequency $f_s$
- pulse duration $T_p$

Choose the waveform:
- rectangular pulse
- LFM chirp (baseband, then take the real part for simplicity)

We also choose a target range $R$ to generate the delay $\tau = 2R/c$ and the sample delay $n_0$.

**Questions**
1. Why do we normalize waveform energy $E_s$ in this lab?
2. What advantage can an LFM waveform have for range estimation?
3. Fill the TODO gaps in the following code

In [4]:
# Radar parameters
c  = 3e8     # [m/s]
fs = 5e6     # [Hz]
Tp = 20e-6   # [s]
N  = int(np.round(Tp*fs))  # samples per pulse

# True target range (used only for simulation)
R_true = 1200.0  # [m]

# TODO: compute round-trip delay tau and delay in samples n0
# tau = ?
# n0  = ? 
raise NotImplementedError("TODO: compute tau and n0")

NotImplementedError: TODO: compute tau and n0

### Waveform generation

Select:
- `waveform="rect"`
- `waveform="lfm"`

Then normalize energy $E_s=\sum s[n]^2$ so that $E_s=1$.

**Task:** Fill the TODO gaps in the following code
1. Make the signal for the rectangular case.
2. Make the signal for the lfm case (find documentation of how to make it, it will depend on some parameters...).
3. Normalize the signal so its energy is 1.

In [5]:
waveform = "lfm"  # "rect" or "lfm"
t = np.arange(N)/fs

# TODO: generate s[n]
# - rect: 
# s = ?

# - lfm: 
# parameters ?
# s = ?
raise NotImplementedError("TODO: generate waveform s")

# TODO: normalize so that Es=1
# s  = ?
raise NotImplementedError("TODO: normalize waveform energy")

plot_signals(s, x1=None, x_axis=None)

NotImplementedError: TODO: generate waveform s

## 2) Data generation (synthetic received signal)

We embed the pulse in a longer record to allow delay search later.

- Under $H_0$: noise only
- Under $H_1$: delayed echo + noise

We parameterize signal-to-noise ratio (SNR) using:

- With $E_s=1$: define $\text{SNR} = A^2/\sigma^2$
- So $A = \sigma\sqrt{\text{SNR}}$.

**Interpretation questions**
1. What does a negative SNR in dB imply about detectability?
2. If you fix $P_{FA}$, does the threshold depend on $A$? Why?
3. Compute A from SNR_dB and sigma (assuming Es=1) Fill the TODO gaps

In [6]:
def simulate_trial(A, sigma, n0, s):
    # Generate one H0 and one H1 record (same length).
    N = len(s)
    L = N + n0 + 50  # buffer length
    w = sigma*np.random.randn(L)

    x0 = w.copy()
    x1 = w.copy()
    x1[n0:n0+N] += A*s
    return x0, x1

In [10]:
SNR_dB = -5.0
sigma = 1.0

# TODO: compute A from SNR_dB and sigma (assuming Es=1)
# A = ?
raise NotImplementedError("TODO: compute A")

x0, x1 = simulate_trial(A, sigma, n0, s)
print("Generated x0, x1 with length:", len(x0))

# Plot both signals
plot_signals(x0, x1, x_axis=None)

NotImplementedError: TODO: compute A

## 3) Matched filter

**Known delay** statistic:

$$
T = \sum_{n=0}^{N-1} x[n_0+n]\,s[n]
$$

**Unknown delay** search:

$$
T[n] = \sum_{k=0}^{N-1} x[n+k]\,s[k], \qquad n\in[0, n_\text{search})
$$

**Tasks** Fill the gaps
1. Function to calculate the T-statistics when the delay is known
2. Function to calculate the T-statistics when the delay is unknown. In this case, there would be an array of possible matches. Plot the array.


In [ ]:
def matched_filter_statistic_known_delay(x, s, n0):
    ??
    
    return T

def matched_filter_search_unknown_delay(x, s, n_search):
    ???
    
    return T_max, n_max, T_All

In [ ]:
# TODO: compute T0 and T1 with known delay
# T0 = 
# T1 = 
raise NotImplementedError("TODO: matched filter statistics known delay")


# TODO: compute T0 and T1 with unknown delay, and plot the array of possible T-statistics
# T0_max, n0_max, T0_All = 
# T1_max, n1_max, T1_All = 
# plt.plot(T0_all,color='r',label="T0")
# plt.plot(T1_all,color='b',label="T1")
# plt.legend()
raise NotImplementedError("TODO: matched filter statistics unknown delay")


## 4) Neyman–Pearson threshold and empirical ROC

Under the Gaussian model (known delay, real matched-filter output):

- Under $H_0$: $T \sim \mathcal{N}(0,\sigma^2 E_s)$
- Under $H_1$: $T \sim \mathcal{N}(A E_s,\sigma^2 E_s)$

Thus,
$$
P_{FA} = Q\left(\frac{\gamma}{\sigma\sqrt{E_s}}\right),
\qquad
\gamma = \sigma\sqrt{E_s}\;Q^{-1}(P_{FA}).
$$

In [ ]:
def threshold_from_pfa(pfa, sigma, Es=1.0):
    return sigma*np.sqrt(Es)*qinv(pfa)

**Tasks** Fill the gaps

1. Function that stimates the ROC using theory (likelihood functions)
2. Function that stimates the ROC using montecarlo with $M=$ number of iterations.
3. Compare and discuss the results.

In [11]:
def estimate_roc_theory(SNR, sigma, Es, pfa_list, M=20000):
    # TODO:
    # - Compute thresholds gamma for each desired pfa
    # - Theory
    raise NotImplementedError("TODO: implement ROC estimation")
    
def estimate_roc_mc(SNR, sigma, s, n0, pfa_list, M=20000):
    # TODO:
    # - Compute thresholds gamma for each desired pfa
    # - Monte Carlo trials to estimate PFA and PD
    raise NotImplementedError("TODO: implement ROC estimation")

pfa_list = np.logspace(-4, -1, 25) # uncomemnt for log plot
pfa_list = np.linspace(0.001, 1, 100) # comment for log plot

Pfa_hat, Pd_hat = estimate_roc_mc(SNR, sigma, s, n0, pfa_list, M=20000)

fig = go.Figure()
fig.add_trace(go.Scatter(x=Pfa_hat, y=Pd_hat, mode='markers+lines', name=f'Empirical ROC (SNR={SNR} dB)'))

# fig.update_layout(xaxis_title='PFA', yaxis_title='PD', xaxis_type='log') # uncomemnt for log plot
fig.update_layout(xaxis_title='PFA', yaxis_title='PD', xaxis_type='linear') # comment for log plot
fig.update_layout(width=400, height=400) # comment for log plot

fig.show()

NameError: name 'A' is not defined

### ROC family for multiple SNR values

**Tasks** Compute ROC curves for multiple SNR values and plot them together.

1. Using theory (likelihood functions)
2. Using montecarlo with $M=$ number of iterations.
3. Compare and discuss the results.

Suggested: `SNR_dB_list = [-10, -5, 0, 5, 10]`

**Note:** Sometimes it is only relevant to plot the results for low PFA (

In [ ]:
SNR_dB_list = [-10, -5, 0, 5, 10] # range of SNR in dB

pfa_list = np.logspace(-4, -1, 25) # uncomemnt for log plot
pfa_list = np.linspace(0.001, 1, 100) # comment for log plot

# TODO: ROC family vs SNR
raise NotImplementedError("TODO: ROC family vs SNR")

# fig.update_layout(xaxis_title='PFA', yaxis_title='PD', xaxis_type='log') # uncomemnt for log plot
fig.update_layout(xaxis_title='PFA', yaxis_title='PD', xaxis_type='linear') # comment for log plot
fig.update_layout(width=400, height=400) # comment for log plot

fig.show()




## 5) Optional: Probability of error $P_e$ and Miss Detecion $P_{MD}$.
We will assumme that the probabilities of target and no target could be different. This will be the priors of each event. In this case, we could estimate the probability of error using this information. Note that in Neyman-Pearson, knowing the priors does not affect $PFA$, $PD$, nor $P_{MD}$. 

With $P(H_1)=\P_1$, $P(H_0)=1-\P_1$:

$$
P_e = (1-\pi_1)P_{FA} + \pi_1 P_{MD}
$$

where $P_{MD}=P(\text{decide }H_0 \mid H_1)$.

**Tasks** 

1. Function that stimates the $P_e$, $PD$ and $P_{MD}$ 
2. Function that stimates the $P_e$, $PD$ and $P_{MD}$ using montecarlo with $M=$ number of iterations.
3. Compute fopr serveral PFA and plot ROCs, $PD$, vs. $P_e$, $PD$, vs. $P_{MD}$. 
4. Compare and discuss the results.

In [ ]:
def estimate_pe_md(SNR, sigma, s, n0, pfa_design=1e-3, prior_H1=P1, M=20000):
    # TODO:
    # - Compute gamma from pfa_design
    # - Estimate Pfa_hat and Pmd_hat with Monte Carlo
    # - Compute Pe_hat
    raise NotImplementedError("TODO: implement Pe estimation")
# P1 = 0.2 # prior probability
Pe_hat, Pfa_hat, Pmd_hat, gamma = estimate_pe_md(SNR, sigma, s, n0, pfa_design=1e-3, prior_H1=P1, M=20000)
print("Pe_hat =", Pe_hat, "Pfa_hat =", Pfa_hat, "Pmd_hat =", Pmd_hat, "gamma =", gamma)

## 6) Unknown delay search and range estimation

We have to estimate:

- $\hat{n}_0 = \arg\max_n T[n])$
- $\hat{\tau} = \hat{n}_0/f_s$
- $\hat{R} = c\hat{\tau}/2$

**Tasks** 

1. Function that stimates the target distance in an specific range
2. Function that stimates the $P_e$, $PD$ and $P_{MD}$ using montecarlo with $M=$ number of iterations.
3. Compute fopr serveral PFA and plot ROCs, $PD$, vs. $P_e$, $PD$, vs. $P_{MD}$. 
4. Compare and discuss the results.

In [ ]:
def estimate_range_from_search(x, s, fs, c, n_search):
    # TODO: use matched_filter_search_unknown_delay, then convert delay to range
    raise NotImplementedError("TODO: implement range estimation")

n_search = n0 + 30 # search range
x0, x1 = simulate_trial(A, sigma, n0, s)

R_hat, n_hat, T_max, T_all = estimate_range_from_search(x1, s, fs, c, n_search)
print("True R =", R_true, "Estimated R =", R_hat, "n_hat =", n_hat)

fig = go.Figure()
fig.add_trace(go.Scatter(y=T_all, mode='lines', name='Matched filter output'))
fig.update_layout(xaxis_title='Delay index n', yaxis_title='Correlation')
fig.show()